# Advanced Gemini Integration with Production-Grade Analytics

**Contributor**: Johan Genis ([@fluxforgeai](https://github.com/fluxforgeai))  
**Based on**: Week 4, Day 3 - Code Generator

## Overview

This notebook extends the Day 3 code generator with three key enhancements:

1. **Gemini Safety Filter Solution** - A 13-approach fallback system that handles Gemini's safety filters blocking complex code conversion requests
2. **Intelligent Cross-Model Fallback** - When one model fails, automatically uses the last successful conversion from another model
3. **Production-Grade Gradio Interface** - Performance comparison table, execution time tracking, and status indicators

### Model Selection

Uses the top 3 frontier models for code generation:

| Model | Provider | Key Strength |
|-------|----------|-------------|
| **Claude Opus 4** | Anthropic | Highest real-repository pass-rate on SWE-Bench |
| **Gemini 2.5 Pro** | Google DeepMind | 1M token context, strong multimodal reasoning |
| **GPT-4.1** | OpenAI | Leading ecosystem integrations, competitive cost |

### The Gemini Safety Filter Problem

Gemini 2.5 Pro's safety filters can block complex code conversion requests, especially for algorithms involving random number generation (LCG), mathematical computations, and array processing. This notebook solves that with a systematic 13-approach prompt variation system that tries different system instruction styles and prompt approaches until one succeeds.

In [ ]:
# imports

import os
import io
import sys
import re
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display
import gradio as gr

In [ ]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY', 'your-google-key-if-not-using-env')

In [ ]:
# initialize clients and models

openai = OpenAI()
claude = anthropic.Anthropic()
google.generativeai.configure(api_key=os.environ['GOOGLE_API_KEY'])

# Frontier models for code generation
OPENAI_MODEL = "gpt-4.1-2025-04-14"
CLAUDE_MODEL = "claude-opus-4-20250514"
GEMINI_MODEL = "gemini-2.5-pro-preview-05-06"

# Want to keep costs ultra-low? Uncomment these lines:
# OPENAI_MODEL = "gpt-4o-mini"
# CLAUDE_MODEL = "claude-3-haiku-20240307"
# GEMINI_MODEL = "gemini-1.5-flash"

In [ ]:
# system message and prompt helpers

system_message = (
    "You are an assistant that reimplements Python code in high performance C++ for an Apple M3 Max. "
    "Respond only with C++ code; use comments sparingly and do not provide any explanation other than occasional comments. "
    "The C++ response needs to produce an identical output in the fastest possible time."
)


def user_prompt_for(python):
    return (
        "Rewrite this Python code in C++ with the fastest possible implementation that produces identical output in the least time. "
        "Respond only with C++ code; do not explain your work other than a few comments. "
        "Pay attention to number types to ensure no int overflows. "
        "Remember to #include all necessary C++ packages such as iomanip.\n\n"
        + python
    )


def messages_for(python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(cpp):
    """Write C++ code to file, extracting from code blocks if needed."""
    # Try to extract code from markdown code blocks
    cpp_blocks = re.findall(r'```cpp\n(.*?)\n```', cpp, re.DOTALL)
    if cpp_blocks:
        code = cpp_blocks[0]
    else:
        code_blocks = re.findall(r'```\n(.*?)\n```', cpp, re.DOTALL)
        if code_blocks:
            code = code_blocks[0]
        else:
            code = cpp.replace("```cpp", "").replace("```", "")
    
    code = code.strip()
    with open("optimized.cpp", "w") as f:
        f.write(code)

## Key Innovation: Gemini Safety Filter Bypass

The `optimize_gemini()` function below is the core innovation. Unlike a simple single-attempt approach, it:

- **Tries 13 different approaches** with varying system instructions and prompt styles
- **Systematically rotates** through academic, tutorial, algorithm-focused, and other prompt styles
- **Provides detailed feedback** on which approach succeeded or why all failed
- **Handles safety filter responses gracefully** with proper error detection

This allows Gemini to successfully process complex algorithms (like LCG random number generators and max subarray sum) that would otherwise be blocked.

In [ ]:
def optimize_gpt(python):
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL, messages=messages_for(python), stream=True
    )
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply)

In [ ]:
def optimize_gemini(python):
    """Try multiple prompt approaches to handle Gemini's safety filters."""
    approaches = [
        {"system_msg": None, "use_system": False, "prompt_style": "gentle"},
        {"system_msg": "You are a helpful coding assistant. Convert Python code to C++.",
         "use_system": True, "prompt_style": "simple"},
        {"system_msg": "Convert Python code to equivalent C++ code. Return only the C++ code.",
         "use_system": True, "prompt_style": "direct"},
        {"system_msg": "You are an academic programming tutor helping with code translation exercises.",
         "use_system": True, "prompt_style": "academic"},
        {"system_msg": "You are a computer science expert specializing in algorithm implementation and optimization.",
         "use_system": True, "prompt_style": "algorithm"},
        {"system_msg": "You help students learn programming by showing equivalent implementations in different languages.",
         "use_system": True, "prompt_style": "tutorial"},
        {"system_msg": "You are a performance optimization specialist. Convert code to faster implementations.",
         "use_system": True, "prompt_style": "performance"},
        {"system_msg": None, "use_system": False, "prompt_style": "minimal"},
        {"system_msg": "You are a software engineer helping with data structure implementations.",
         "use_system": True, "prompt_style": "datastructures"},
        {"system_msg": "You help with mathematical computations and numerical algorithms.",
         "use_system": True, "prompt_style": "mathematical"},
        {"system_msg": "You are reviewing code and suggesting optimized implementations.",
         "use_system": True, "prompt_style": "codereview"},
        {"system_msg": "You help port code between different programming languages for compatibility.",
         "use_system": True, "prompt_style": "porting"},
        {"system_msg": None, "use_system": False, "prompt_style": "simplified"},
    ]

    prompt_templates = {
        "gentle": "Hello! Could you help me translate this Python code to C++? Please include the necessary headers:\n\n{python}",
        "simple": "Convert this Python code to C++:\n\n{python}",
        "direct": "Translate to C++ with headers:\n\n{python}",
        "academic": "For this programming exercise, please convert the following Python code to equivalent C++ code:\n\n{python}",
        "algorithm": "I need to implement these algorithms in C++ for better performance. Here's the Python implementation:\n\n{python}\n\nPlease provide the equivalent C++ implementation with appropriate headers.",
        "tutorial": "As a learning exercise, let's see how this Python code would look in C++. Here's the Python version:\n\n{python}\n\nCould you show the C++ equivalent?",
        "performance": "I need to optimize this Python code by converting it to C++. Here's the current implementation:\n\n{python}\n\nPlease provide a high-performance C++ version.",
        "minimal": "C++ version:\n\n{python}",
        "datastructures": "I need to implement these data structures and algorithms in C++. Here's the current Python code:\n\n{python}\n\nCould you provide the C++ implementation?",
        "mathematical": "Help me convert this mathematical computation from Python to C++:\n\n{python}\n\nPlease provide the C++ equivalent with proper headers.",
        "codereview": "During code review, we decided to rewrite this Python code in C++ for better performance:\n\n{python}\n\nWhat would the C++ version look like?",
        "porting": "I'm porting this Python library to C++ for cross-platform compatibility:\n\n{python}\n\nPlease show the equivalent C++ implementation.",
        "simplified": "Please help convert this to C++:\n\n{python}",
    }

    for i, approach in enumerate(approaches):
        try:
            print(f"Trying approach {i+1}...")

            if approach["use_system"] and approach["system_msg"]:
                model = google.generativeai.GenerativeModel(
                    GEMINI_MODEL, system_instruction=approach["system_msg"]
                )
            else:
                model = google.generativeai.GenerativeModel(GEMINI_MODEL)

            prompt = prompt_templates[approach["prompt_style"]].format(python=python)

            response = model.generate_content(
                prompt,
                generation_config=google.generativeai.types.GenerationConfig(
                    max_output_tokens=4000, temperature=0.1,
                ),
                safety_settings=[
                    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
                ],
            )

            if hasattr(response, 'candidates') and response.candidates:
                candidate = response.candidates[0]
                if hasattr(candidate, 'finish_reason'):
                    if candidate.finish_reason == 2:  # SAFETY
                        print(f"Approach {i+1} blocked by safety filters.")
                        continue
                    elif candidate.finish_reason == 3:  # RECITATION
                        print(f"Approach {i+1} blocked due to recitation concerns.")
                        continue

                if (hasattr(candidate, 'content') and hasattr(candidate.content, 'parts')
                        and candidate.content.parts):
                    reply = "".join(
                        part.text for part in candidate.content.parts if hasattr(part, 'text')
                    )
                    if reply.strip():
                        print(f"Success with approach {i+1}!")
                        print(reply)
                        write_output(reply)
                        return
                    else:
                        print(f"Approach {i+1} returned empty content.")
                        continue
                else:
                    print(f"Approach {i+1} returned no content parts.")
                    continue
            else:
                print(f"Approach {i+1} returned no candidates.")
                continue

        except Exception as e:
            print(f"Approach {i+1} failed with error: {e}")
            continue

    print("All approaches failed. Gemini safety filters are being very strict with this content.")

In [ ]:
def optimize_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

## Test 1: Simple Algorithm (Pi Calculation)

All models should handle this without issues.

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

exec(pi)

In [ ]:
optimize_gpt(pi)

In [ ]:
# Compile and run - adjust the compile command for your platform

!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

In [ ]:
optimize_claude(pi)

In [ ]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

In [ ]:
optimize_gemini(pi)

In [ ]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

## Test 2: Complex Algorithm (LCG + Max Subarray Sum)

This is where Gemini's safety filters often activate. The LCG random number generator and array processing patterns can trigger safety blocks. Watch the `optimize_gemini()` function cycle through its 13 approaches.

In [ ]:
python_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

exec(python_hard)

In [ ]:
optimize_gpt(python_hard)

In [ ]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

In [ ]:
optimize_claude(python_hard)

In [ ]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

In [ ]:
optimize_gemini(python_hard)

In [ ]:
!clang++ -O3 -std=c++17 -march=armv8.3-a -o optimized optimized.cpp
!./optimized

## Production-Grade Gradio Interface

The interface below combines all enhancements into a unified application:

- **Convert code** with any of the three models
- **Automatic fallback** when Gemini's safety filters activate
- **Performance comparison table** tracking Python vs C++ execution times per model
- **Status indicators** showing when fallback is in use

In [ ]:
# Streaming versions for the Gradio interface

def stream_gpt(python):
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL, messages=messages_for(python), stream=True
    )
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```cpp\n', '').replace('```', '')


def stream_gemini(python):
    model = google.generativeai.GenerativeModel(
        GEMINI_MODEL, system_instruction=system_message
    )
    try:
        response = model.generate_content(
            user_prompt_for(python),
            generation_config=google.generativeai.types.GenerationConfig(
                max_output_tokens=4000, temperature=0.1,
            ),
            safety_settings=[
                {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
            ],
            stream=True,
        )
        reply = ""
        has_content = False
        for chunk in response:
            if hasattr(chunk, 'candidates') and chunk.candidates:
                candidate = chunk.candidates[0]
                if hasattr(candidate, 'finish_reason'):
                    if candidate.finish_reason == 2:
                        yield "Gemini blocked the response due to safety filters."
                        return
                    elif candidate.finish_reason == 3:
                        yield "Gemini blocked the response due to recitation concerns."
                        return
                if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                    for part in candidate.content.parts:
                        if hasattr(part, 'text') and part.text:
                            reply += part.text
                            has_content = True
                            yield reply.replace('```cpp\n', '').replace('```', '')
        if not has_content:
            yield "Gemini returned an empty response."
    except Exception as e:
        yield f"Error: {e}"


def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```cpp\n', '').replace('```', '')


def optimize(python, model):
    if model == "GPT":
        result = stream_gpt(python)
    elif model == "Claude":
        result = stream_claude(python)
    elif model == "Gemini":
        result = stream_gemini(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far

In [ ]:
# Execution and analytics helpers

def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()


def execute_cpp(code):
    """Compile and run C++ code. Adjust compile_cmd for your platform."""
    write_output(code)
    try:
        compile_cmd = [
            "clang++", "-Ofast", "-std=c++17",
            "-march=armv8.5-a", "-mtune=apple-m3", "-mcpu=apple-m3",
            "-o", "optimized", "optimized.cpp"
        ]
        subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        run_result = subprocess.run(
            ["./optimized"], check=True, text=True, capture_output=True
        )
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"


def extract_execution_time(output):
    match = re.search(r'Execution Time: ([\d.]+) seconds', output)
    if match:
        return f"{float(match.group(1)):.6f}s"
    return ""


def is_conversion_successful(output):
    error_indicators = [
        "safety filters", "All approaches failed", "blocked the response",
        "An error occurred", "Error:", "Gemini blocked", "Response blocked"
    ]
    return not any(ind in output for ind in error_indicators) and output.strip() != ""

In [ ]:
# Fallback and results tracking

def handle_model_conversion(python_code, model_name, last_successful_code, last_successful_model):
    """Convert code with automatic fallback to last successful result."""
    try:
        final_result = ""
        for partial_result in optimize(python_code, model_name):
            final_result = partial_result

        if is_conversion_successful(final_result):
            return final_result, final_result, model_name, False, {}
        else:
            if last_successful_code.strip():
                fallback_message = (
                    f"// {model_name} conversion failed, using {last_successful_model} conversion\n\n"
                    + last_successful_code
                )
                fallback_info = {"failed_model": model_name, "source_model": last_successful_model}
                return fallback_message, last_successful_code, last_successful_model, True, fallback_info
            else:
                return final_result, last_successful_code, last_successful_model, False, {}
    except Exception as e:
        if last_successful_code.strip():
            fallback_message = (
                f"// {model_name} conversion error, using {last_successful_model} conversion\n\n"
                + last_successful_code
            )
            fallback_info = {"failed_model": model_name, "source_model": last_successful_model}
            return fallback_message, last_successful_code, last_successful_model, True, fallback_info
        else:
            return f"Error: {str(e)}", last_successful_code, last_successful_model, False, {}


def update_python_results(python_output, model_name, current_results):
    python_time = extract_execution_time(python_output)
    updated_results = []
    for row in current_results:
        if row[0] == model_name:
            updated_results.append([model_name, python_time, row[2]])
        else:
            updated_results.append(row)
    return python_output, updated_results


def update_cpp_results(cpp_output, model_name, current_results, is_fallback_active, fallback_details):
    error_indicators = [
        "All approaches failed", "safety filters", "An error occurred",
        "Error:", "blocked the response"
    ]
    if any(ind in cpp_output for ind in error_indicators):
        cpp_time = "Error"
    else:
        cpp_time = extract_execution_time(cpp_output)

    comparison_msg = ""
    if is_fallback_active and fallback_details and cpp_time != "Error":
        failed_model = fallback_details.get("failed_model", "")
        source_model = fallback_details.get("source_model", "")
        original_time = None
        for row in current_results:
            if row[0] == source_model and row[2] and "s" in row[2]:
                try:
                    original_time = float(row[2].replace("s", ""))
                    break
                except ValueError:
                    pass
        if original_time and cpp_time and "s" in cpp_time:
            try:
                current_time = float(cpp_time.replace("s", ""))
                time_diff = abs(current_time - original_time)
                faster_slower = "faster" if current_time < original_time else "slower"
                comparison_msg = (
                    f"\n\nPerformance: {failed_model} used {source_model}'s code. "
                    f"C++ time was {cpp_time} ({faster_slower} by {time_diff:.6f}s "
                    f"vs {source_model}'s {original_time:.6f}s)"
                )
            except ValueError:
                pass

    updated_results = []
    for row in current_results:
        if row[0] == model_name:
            updated_results.append([model_name, row[1], cpp_time])
        else:
            updated_results.append(row)

    final_output = cpp_output + comparison_msg if comparison_msg else cpp_output
    return final_output, updated_results

In [ ]:
# Launch the Gradio interface

css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
.status-message {
    padding: 10px;
    border-radius: 5px;
    margin: 10px 0;
    border-left: 4px solid #ff6b6b;
    background-color: #ffe0e0;
}
"""


def convert_with_fallback(python_code, model_name, last_code, last_model):
    cpp_result, updated_code, updated_model, is_fallback, fb_details = handle_model_conversion(
        python_code, model_name, last_code, last_model
    )
    if cpp_result.startswith(f"// {model_name} conversion failed"):
        source = fb_details.get("source_model", "previous model")
        status = gr.update(value=f"Warning: {model_name} conversion failed - using fallback from {source}", visible=True)
    elif cpp_result.startswith(f"// {model_name} conversion error"):
        source = fb_details.get("source_model", "previous model")
        status = gr.update(value=f"Error: {model_name} conversion error - using fallback from {source}", visible=True)
    else:
        status = gr.update(value="", visible=False)
    return cpp_result, updated_code, updated_model, is_fallback, fb_details, status


with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to C++")

    results_state = gr.State([["GPT", "", ""], ["Claude", "", ""], ["Gemini", "", ""]])
    last_successful_code = gr.State("")
    last_successful_model = gr.State("")
    is_using_fallback = gr.State(False)
    fallback_info = gr.State({"failed_model": "", "source_model": ""})

    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")
    with gr.Row():
        convert = gr.Button("Convert code")

    status_msg = gr.Markdown("", visible=False, elem_classes=["status-message"])

    with gr.Row():
        python_run = gr.Button("Run Python")
        cpp_run = gr.Button("Run C++")
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="C++ result:", elem_classes=["cpp"])

    gr.Markdown("## Performance Comparison")
    results_table = gr.Dataframe(
        headers=["Model", "Python Time", "C++ Time"],
        datatype=["str", "str", "str"],
        value=[["GPT", "", ""], ["Claude", "", ""], ["Gemini", "", ""]],
        label="Execution Times",
        interactive=False,
        wrap=True,
    )

    convert.click(
        convert_with_fallback,
        inputs=[python, model, last_successful_code, last_successful_model],
        outputs=[cpp, last_successful_code, last_successful_model, is_using_fallback, fallback_info, status_msg],
    )
    python_run.click(execute_python, inputs=[python], outputs=[python_out]).then(
        update_python_results, inputs=[python_out, model, results_state], outputs=[python_out, results_state]
    ).then(lambda results: results, inputs=[results_state], outputs=[results_table])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out]).then(
        update_cpp_results,
        inputs=[cpp_out, model, results_state, is_using_fallback, fallback_info],
        outputs=[cpp_out, results_state],
    ).then(lambda results: results, inputs=[results_state], outputs=[results_table])

ui.launch(inbrowser=True)